# GDR-Net++ Baseline Evaluation -- Colab GPU

**Ejecutar DESPUES de `00_colab_setup.ipynb`**

Este notebook:
1. Instala GDR-Net++ (gdrnpp_bop2022)
2. Descarga pesos pre-entrenados (BOP 2022 winner)
3. Ejecuta inferencia en YCB-V y T-LESS
4. Calcula metricas BOP y compara con FoundationPose

**Requiere:** GPU T4 o superior

In [ ]:
import torch, os, sys
assert torch.cuda.is_available(), "GPU requerida!"
print(f"GPU: {torch.cuda.get_device_name(0)}")

REPO_DIR = "/content/repo_tfm"
assert os.path.exists(REPO_DIR), "Ejecuta 00_colab_setup.ipynb primero"
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

## 1. Instalar GDR-Net++

In [ ]:
GDRNET_DIR = "/content/gdrnpp_bop2022"

if not os.path.exists(GDRNET_DIR):
    !git clone https://github.com/shanice-l/gdrnpp_bop2022.git {GDRNET_DIR}
    print("GDR-Net++ clonado")
else:
    print("GDR-Net++ ya existe")

# Instalar dependencias
!cd {GDRNET_DIR} && pip install -q -e . 2>/dev/null || echo "Instalacion parcial (algunos deps opcionales)"
!pip install -q mmcv-full mmdet 2>/dev/null || echo "mmcv/mmdet may need manual install"

print("\nGDR-Net++ listo")

In [ ]:
# Descargar pesos pre-entrenados
# Ref: https://github.com/shanice-l/gdrnpp_bop2022#model-zoo
USE_GDRIVE = os.path.exists('/content/drive/MyDrive')
WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights/gdrnet" if USE_GDRIVE else "/content/weights/gdrnet"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# GDR-Net++ model zoo URLs (BOP 2022 challenge models)
GDRNET_MODELS = {
    "ycbv": {
        "url": "https://dl.fbaipublicfiles.com/gdrnpp/ycbv/gdrn_ycbv.pth",
        "config": "configs/gdrn/ycbv/convnext_a6_AugCosyAAEGray_BG05_mlL1_DMask_amodalClipBox_classaware_ycbv.py",
    },
    "tless": {
        "url": "https://dl.fbaipublicfiles.com/gdrnpp/tless/gdrn_tless.pth",
        "config": "configs/gdrn/tless/convnext_a6_AugCosyAAEGray_BG05_mlL1_DMask_amodalClipBox_classaware_tless.py",
    },
}

print("GDR-Net++ pesos:")
print("  Nota: Los pesos exactos dependen de la version del challenge.")
print("  Consulta: https://github.com/shanice-l/gdrnpp_bop2022#model-zoo")
print(f"  Directorio: {WEIGHTS_DIR}")

# Intentar descargar automaticamente
for dataset, info in GDRNET_MODELS.items():
    weight_path = f"{WEIGHTS_DIR}/gdrn_{dataset}.pth"
    if not os.path.exists(weight_path):
        print(f"\nDescargando pesos {dataset}...")
        !wget -q --show-progress -O {weight_path} "{info['url']}" 2>&1 || echo "Descarga manual necesaria"
    else:
        print(f"[OK] {dataset} pesos ya descargados")

## 2. Ejecutar GDR-Net++ en YCB-V

In [ ]:
import numpy as np
import json
import time
from pathlib import Path
from tqdm.notebook import tqdm

from src.perception.gdrnet import GDRNetEstimator
from src.utils.dataset_loader import BOPDataset

DATA_DIR = f"{REPO_DIR}/data/datasets"

# Cargar dataset
ycbv = BOPDataset(f"{DATA_DIR}/ycbv", split="test")
scenes = ycbv.get_scene_ids()
print(f"YCB-V: {len(scenes)} escenas")

# Inicializar GDR-Net
gdrnet = GDRNetEstimator(
    weights_path=f"{WEIGHTS_DIR}/gdrn_ycbv.pth",
    device="cuda",
)

In [ ]:
# Ejecutar inferencia
results_gdrnet_ycbv = []
timing_total = 0
n_evaluated = 0

MAX_SCENES = 5  # Limitar para primera prueba
eval_scenes = scenes[:MAX_SCENES] if MAX_SCENES else scenes

for scene_id in tqdm(eval_scenes, desc="GDR-Net++ YCB-V"):
    n_images = ycbv.get_num_images(scene_id)
    gt_poses = ycbv.load_scene_gt(scene_id)
    
    for img_id in range(min(n_images, 50)):
        try:
            sample = ycbv.load_sample(scene_id, img_id)
            gt_list = sample.get('gt_poses', [])
            
            for gt in gt_list:
                t0 = time.time()
                pred = gdrnet.estimate_pose(
                    rgb=sample['rgb'],
                    bbox=None,  # Use GT bbox or detected bbox
                    K=sample['cam_K'],
                    obj_id=gt['obj_id'],
                )
                elapsed = time.time() - t0
                timing_total += elapsed
                n_evaluated += 1
                
                results_gdrnet_ycbv.append({
                    'scene_id': scene_id,
                    'img_id': img_id,
                    'obj_id': gt['obj_id'],
                    'R_pred': pred['R'].tolist(),
                    't_pred': pred['t'].tolist(),
                    'score': pred['score'],
                    'time_s': elapsed,
                })
        except Exception as e:
            pass  # Skip failures silently for batch eval

avg_time = timing_total / max(n_evaluated, 1)
print(f"\nGDR-Net++ evaluadas: {n_evaluated} objetos")
print(f"Tiempo promedio: {avg_time*1000:.1f} ms/objeto")

## 3. Comparar FoundationPose vs GDR-Net++

In [ ]:
from src.perception.evaluator import PoseEvaluator
import matplotlib.pyplot as plt

models_dir = f"{DATA_DIR}/ycbv/models"
evaluator = PoseEvaluator(
    dataset_root=f"{DATA_DIR}/ycbv",
    models_dir=models_dir,
)

# Evaluar GDR-Net++
metrics_gdrnet = evaluator.evaluate(
    predictions=results_gdrnet_ycbv,
    metrics=['add', 'adds', 'mssd', 'mspd'],
)
print("GDR-Net++ metrics:")
for k, v in metrics_gdrnet.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# Cargar resultados de FoundationPose (del notebook anterior)
fp_results_dir = "/content/drive/MyDrive/TFM/experiments/foundationpose_ycbv" if USE_GDRIVE else f"{REPO_DIR}/experiments/foundationpose_ycbv"
fp_files = sorted(Path(fp_results_dir).glob("results_*.json")) if os.path.exists(fp_results_dir) else []

if fp_files:
    with open(fp_files[-1]) as f:
        fp_data = json.load(f)
    
    metrics_fp = evaluator.evaluate(
        predictions=fp_data.get('predictions', []),
        metrics=['add', 'adds', 'mssd', 'mspd'],
    )
    print("\nFoundationPose metrics:")
    for k, v in metrics_fp.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
else:
    print("\nFoundationPose results no encontrados.")
    print("Ejecuta 01_foundationpose_eval.ipynb primero.")
    metrics_fp = None

In [ ]:
# Tabla comparativa
print("\n" + "=" * 70)
print("  COMPARATIVA YCB-Video: FoundationPose vs GDR-Net++")
print("=" * 70)

metric_names = ['add_mean', 'adds_mean', 'mssd_auc', 'mspd_auc']
display_names = ['ADD (mm)', 'ADD-S (mm)', 'MSSD AUC', 'MSPD AUC']

print(f"  {'Metrica':<15} {'FoundationPose':>15} {'GDR-Net++':>15} {'Mejor':>10}")
print(f"  {'-'*55}")

for name, display in zip(metric_names, display_names):
    fp_val = metrics_fp.get(name, 0.0) if metrics_fp else 0.0
    gdr_val = metrics_gdrnet.get(name, 0.0)
    
    # For AUC higher is better, for distances lower is better
    if 'auc' in name:
        better = 'FP' if fp_val >= gdr_val else 'GDR'
    else:
        better = 'FP' if fp_val <= gdr_val else 'GDR'
    
    print(f"  {display:<15} {fp_val:>15.4f} {gdr_val:>15.4f} {better:>10}")

print()

In [ ]:
# Grafico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart de metricas
auc_metrics = ['mssd_auc', 'mspd_auc']
labels = ['MSSD AUC', 'MSPD AUC']
fp_vals = [metrics_fp.get(m, 0) for m in auc_metrics] if metrics_fp else [0, 0]
gdr_vals = [metrics_gdrnet.get(m, 0) for m in auc_metrics]

x = np.arange(len(labels))
width = 0.35
axes[0].bar(x - width/2, fp_vals, width, label='FoundationPose', color='#0098CD')
axes[0].bar(x + width/2, gdr_vals, width, label='GDR-Net++', color='#FF6B35')
axes[0].set_ylabel('AUC (higher is better)')
axes[0].set_title('YCB-V: AUC Comparison')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Distance metrics
dist_metrics = ['add_mean', 'adds_mean']
labels2 = ['ADD', 'ADD-S']
fp_vals2 = [metrics_fp.get(m, 0) for m in dist_metrics] if metrics_fp else [0, 0]
gdr_vals2 = [metrics_gdrnet.get(m, 0) for m in dist_metrics]

x2 = np.arange(len(labels2))
axes[1].bar(x2 - width/2, fp_vals2, width, label='FoundationPose', color='#0098CD')
axes[1].bar(x2 + width/2, gdr_vals2, width, label='GDR-Net++', color='#FF6B35')
axes[1].set_ylabel('Distance mm (lower is better)')
axes[1].set_title('YCB-V: Distance Comparison')
axes[1].set_xticks(x2); axes[1].set_xticklabels(labels2)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('FoundationPose vs GDR-Net++ -- YCB-Video', fontsize=14)
plt.tight_layout()

# Guardar
fig_dir = "/content/drive/MyDrive/TFM/experiments/comparison" if USE_GDRIVE else f"{REPO_DIR}/experiments/comparison"
os.makedirs(fig_dir, exist_ok=True)
fig.savefig(f"{fig_dir}/fp_vs_gdrnet_ycbv.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Figura guardada: {fig_dir}/fp_vs_gdrnet_ycbv.png")

## 4. Guardar resultados

In [ ]:
from datetime import datetime

base_dir = "/content/drive/MyDrive/TFM/experiments" if USE_GDRIVE else f"{REPO_DIR}/experiments"
exp_dir = f"{base_dir}/gdrnet_ycbv"
os.makedirs(exp_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

output = {
    'method': 'GDR-Net++',
    'dataset': 'ycbv',
    'n_objects': n_evaluated,
    'avg_time_ms': avg_time * 1000,
    'gpu': torch.cuda.get_device_name(0),
    'metrics': {k: v for k, v in metrics_gdrnet.items() if isinstance(v, (int, float))},
    'predictions': results_gdrnet_ycbv,
}

result_file = f"{exp_dir}/results_{timestamp}.json"
with open(result_file, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Resultados guardados: {result_file}")

# Guardar comparativa
comparison = {
    'dataset': 'ycbv',
    'timestamp': timestamp,
    'foundationpose': {k: v for k, v in (metrics_fp or {}).items() if isinstance(v, (int, float))},
    'gdrnet': {k: v for k, v in metrics_gdrnet.items() if isinstance(v, (int, float))},
}
comp_dir = f"{base_dir}/comparison"
os.makedirs(comp_dir, exist_ok=True)
with open(f"{comp_dir}/comparison_ycbv_{timestamp}.json", 'w') as f:
    json.dump(comparison, f, indent=2)

print(f"Comparativa guardada: {comp_dir}/comparison_ycbv_{timestamp}.json")
print("\nSiguiente: usa estos resultados para el Capitulo 6 de la memoria.")